# VOICE-CUE sLLM 파인튜닝 — Kaggle (백그라운드 실행)

기획서 4-다② "폐쇄망 구동이 가능한 오픈소스 한국어 특화 sLLM을 LoRA로 학습" 단계를 실행합니다.

**Colab 대신 Kaggle을 쓰는 이유**: Colab은 인터랙티브 세션이 브라우저 연결에 묶여 있어 자꾸
끊깁니다(Pro+의 "백그라운드 실행"도 2026년 들어 불안정하다는 신고가 많습니다). Kaggle은
"Save Version"을 누르면 그 시점 노트북 전체가 **별도 머신에서 완전히 분리되어** 처음부터
끝까지 자동 실행됩니다 — 브라우저를 닫아도, 컴퓨터를 꺼도 계속 돕니다.

**시작 전에 반드시 (우측 패널 Settings)**:
1. **Accelerator** → GPU P100 (또는 GPU T4 x2)
2. **Internet** → On (꺼져 있으면 GitHub 클론·pip install이 실패합니다)

**실행 방법 — 이 노트북은 두 단계로 씁니다.**
1. 먼저 1~5번 셀을 인터랙티브하게(Shift+Enter) 하나씩 실행해 GPU·설치·데이터·사전 점검까지
   문제없는지 확인합니다. 여기까지는 수 분이면 끝나고, 인터랙티브 세션이 끊겨도 상관없습니다.
2. 문제가 없으면 우측 상단 **Save Version → Save & Run All (Commit)** 을 누릅니다. 그 순간부터
   창을 닫아도 6~9번 셀(베이스라인 → 학습 → 평가 → 병합)이 별도 머신에서 끝까지 돌아갑니다.
   진행 상황은 나중에 아무 때나 들어가서 그 버전의 로그로 확인하면 됩니다.

무료 할당량은 주당 약 30 GPU시간, 세션(커밋 실행 포함)당 최대 약 12시간입니다. 3번 셀 기본값
(1.5B/2에폭, 약 2.4시간)은 여유 있게 들어갑니다.

## 1. GPU 확인

None이 나오면 우측 Settings 패널에서 Accelerator를 GPU로 바꾸고 세션을 다시 시작하세요.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

import subprocess, sys
out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
if not out.strip():
    sys.exit("GPU가 없습니다. 우측 Settings 패널 > Accelerator에서 GPU(P100 또는 T4 x2)를 선택하세요.")
print(out)

## 2. 저장소 클론 + 의존성 설치

Internet이 꺼져 있으면 이 셀에서 git clone부터 실패합니다 — Settings 패널에서 켜져 있는지 먼저 확인하세요.
설치에 3~5분 걸립니다.

In [ ]:
%cd /kaggle/working
!rm -rf voiceq-air3
!git clone --branch feature/llm-cop-layout --depth 1 https://github.com/MeinBau/voiceq-air3.git
%cd /kaggle/working/voiceq-air3

!pip install -q -r finetune/requirements-train.txt

import importlib
for mod in ("torch", "transformers", "trl", "peft", "bitsandbytes", "datasets", "accelerate"):
    try:
        print(f"{mod:14s} {importlib.import_module(mod).__version__}")
    except Exception as e:
        print(f"{mod:14s} 로드 실패: {e}")

## 3. 학습 설정

1.5B / 1에폭으로 파이프라인이 완주하는 것을 이미 확인했고(상황유형 정확도 45%→75%),
지금 기본값은 **기획서 목표(90%)에 도전하는 3B / 2에폭**입니다.

T4 한 장 기준 메모리는 약 5.2GB로 여유가 있습니다 — Qwen2.5는 어휘가 151,936이라
logits(약 3GB)가 사용량을 지배하고, 이 값은 모델 크기가 아니라 배치·길이로 정해지기
때문에 1.5B에서 3B로 올려도 크게 늘지 않습니다. 그래서 `BATCH_SIZE`는 올리지 마세요 —
거기가 메모리를 실제로 밀어 올리는 손잡이입니다.

먼저 파이프라인만 빠르게 확인하고 싶으면 1.5B / 1에폭으로 되돌리면 33분이면 끝납니다.

| 모델 | 4bit 크기 | 1 에폭 | 2 에폭 | 비고 |
|---|---|---|---|---|
| `Qwen/Qwen2.5-1.5B-Instruct` | ~1.1GB | 33분 (실측) | 약 1.1h | 파이프라인 확인용 |
| `Qwen/Qwen2.5-3B-Instruct` | ~2.0GB | 약 1.1h | **약 2.2h (기본값)** | 기획서 3-다 "2~4GB"에 부합 |

위는 GPU 2장(Kaggle T4 x2, DDP) 기준이고, 1.5B/1에폭 33분 실측에서 환산한 값이다.
평가·병합까지 더하면 전체는 3시간 안팎이며, Kaggle 세션 한도(약 12시간) 안에 든다.
GPU가 한 장이면 학습 시간은 대략 2배가 된다.

기획서 3-다의 "4bit 2~4GB" 목표에 정확히 맞는 것은 3B다. 3B/3에폭(약 7.2시간)도 Kaggle GPU
세션 한도(약 12시간) 안에 들어오므로, 1.5B로 파이프라인을 먼저 완주시킨 뒤 최종 수치는 3B로
다시 커밋 실행하는 순서를 권장한다.

체크포인트·어댑터·병합본은 모두 `/kaggle/working` 아래에 저장되므로, Save & Run All(Commit)이
끝나면 그 버전의 Output 탭에서 자동으로 받을 수 있다(Colab처럼 Drive 마운트가 필요 없다).

In [ ]:
MODEL = "Qwen/Qwen2.5-3B-Instruct"     # 빠르게 확인만 할 때는: "Qwen/Qwen2.5-1.5B-Instruct"
EPOCHS = 2
BATCH_SIZE = 2
GRAD_ACCUM = 8      # 실효 배치 = BATCH_SIZE * GRAD_ACCUM = 16
MAX_SEQ_LEN = 2048  # 데이터 p99가 1699라 잘림 없음

# 평가에 쓸 test 턴 수(전체 179턴). 턴마다 FAST/FULL 두 번 생성하므로 여기가
# 평가 시간을 좌우한다. 40턴이면 3B 기준 베이스라인·튜닝 합쳐 1시간 안팎이다.
# 40턴은 표본이 작아 정확도 한 건이 2.5%p씩 움직인다 — 최종 수치로 인용할 때는
# 179(전체)로 올리는 편이 낫지만 평가에만 4시간 넘게 걸린다.
EVAL_LIMIT = 40

# 화면 배치를 모델이 직접 내게 학습할지(기획서 원안 구조). False면 지금까지와 같이
# 상황 유형만 학습하고 배치는 플레이북(코드)이 정한다. 자세한 배경과 대가(출력 토큰
# +153%)는 finetune/README.md 8.7절 참고.
LAYOUT = False

DATA_DIR = "finetune/data_layout" if LAYOUT else "finetune/data"
LAYOUT_FLAG = "--layout" if LAYOUT else ""
print("학습 구조:", "모델이 화면 배치까지 출력 (원안)" if LAYOUT else "상황 유형만 (현행)")

# 학습이 끝나면 병합본을 이 저장소로 자동 업로드한다(10번 셀). 본인 계정 이름으로
# 바꾸고, 이미 만들어 둔 저장소가 있으면 그 이름을 그대로 써도 된다(같은 저장소에
# 새 커밋으로 덮어쓴다).
HF_REPO_ID = "Jun13KU/voicecue-qwen2.5-3b" + ("-layout" if LAYOUT else "")

import os
OUT = "/kaggle/working/voiceq-air3/finetune/out"
os.makedirs(OUT, exist_ok=True)
print("체크포인트 경로:", OUT)

## 4. 학습 데이터 생성

데이터는 저장소에 커밋되어 있지 않고 시드 고정 생성물입니다(`--seed 20260829` 기본값).
여기서 만든 것과 로컬에서 만든 것이 완전히 같습니다.

In [ ]:
%cd /kaggle/working/voiceq-air3
!python finetune/gen_dataset.py {"--layout-target --out " + DATA_DIR if LAYOUT else ""}
assert _exit_code == 0, f"데이터 생성 실패 (exit {_exit_code}) — 위 트레이스백을 확인하세요."

## 5. 학습 전 점검

여기까지가 "인터랙티브하게 눈으로 확인하는" 구간입니다. 학습을 몇 시간 돌린 뒤에 데이터가
깨져 있던 걸 발견하면 그 시간을 통째로 버립니다. 스키마·토큰 길이와 평가 지표 계산식이
정상인지 먼저 확인합니다(gold는 100%가 나와야 정상). **여기까지 이상 없으면 Save & Run All로
넘어가도 됩니다.**

In [ ]:
!python finetune/train_lora.py --dry-run --model $MODEL --max-seq-len $MAX_SEQ_LEN --data $DATA_DIR
assert _exit_code == 0, f"dry-run 실패 (exit {_exit_code})"
!python tests/test_tiling.py
assert _exit_code == 0, f"타일링 불변식 테스트 실패 (exit {_exit_code})"
!python finetune/evaluate.py --backend gold --split test --data $DATA_DIR $LAYOUT_FLAG
assert _exit_code == 0, f"평가 하네스 자기검증 실패 (exit {_exit_code}) — gold인데 100%가 아니면 지표 코드가 깨진 것"

## 6. 베이스라인 측정 (튜닝 전)

여기부터는 시간이 걸리므로 **Save & Run All(Commit)로 백그라운드 실행하는 것을 권장**합니다.

파인튜닝의 효과를 말하려면 비교 대상이 있어야 합니다. **튜닝 전 모델에는 few-shot을 켜서**
측정합니다 — few-shot 없이 형식을 지키게 만드는 것 자체가 파인튜닝의 성과이므로, 양쪽에
똑같이 켜면 그 효과가 측정되지 않습니다.

전체 test 179턴 × 2경로는 시간이 걸리므로 `--limit 40`으로 표본만 봅니다.

In [ ]:
!python finetune/evaluate.py --backend hf --model $MODEL \
    --split test --limit $EVAL_LIMIT --few-shot --data $DATA_DIR $LAYOUT_FLAG \
    --out $OUT/baseline.json
assert _exit_code == 0, f"베이스라인 평가 실패 (exit {_exit_code}) — 위 트레이스백을 확인하세요."

## 7. QLoRA 학습

여기가 가장 오래 걸리는 부분입니다. 10 스텝마다 loss가 찍히고, 에폭마다 체크포인트와 평가
loss가 남습니다.

GPU가 두 장이면(Kaggle T4 x2) 자동으로 데이터 병렬(DDP)로 두 장을 다 씁니다 — 한 장만 쓸
때보다 거의 두 배 빠릅니다. 실효 배치는 GPU 장수와 무관하게 같게 유지됩니다.

`--save-steps` 주기(기본 25스텝)마다 체크포인트가 남습니다. 도중에 죽거나 끊겨도 이 셀을
다시 실행하면 마지막 체크포인트를 찾아 이어서 학습하므로, 진행분이 통째로 날아가지 않습니다.

다만 인터랙티브 세션이 완전히 종료되면 `/kaggle/working` 내용도 같이 사라집니다. 정말
끊김 없이 돌리려면 위 안내대로 **Save & Run All(Commit)**을 쓰세요.

In [ ]:
import torch

# GPU가 여러 장이면 데이터 병렬(DDP)로 전부 쓴다. Kaggle의 T4 x2가 여기 해당하며,
# 거의 장수에 비례해 빨라진다. 한 장뿐이면(Colab T4, Kaggle P100) 그냥 python으로 띄운다.
N_GPU = torch.cuda.device_count()
# 실효 배치(BATCH_SIZE x GRAD_ACCUM x 프로세스 수)를 GPU 장수와 무관하게 고정한다.
ACCUM = max(1, GRAD_ACCUM // max(1, N_GPU))
LAUNCH = (f"accelerate launch --num_processes {N_GPU} --mixed_precision fp16"
          if N_GPU > 1 else "python")
print(f"GPU {N_GPU}장 · 실행 방식: {LAUNCH.split()[0]} · "
      f"실효 배치 {BATCH_SIZE}x{ACCUM}x{max(1, N_GPU)} = {BATCH_SIZE * ACCUM * max(1, N_GPU)}")

!{LAUNCH} finetune/train_lora.py \
    --model $MODEL \
    --out $OUT \
    --epochs $EPOCHS \
    --batch-size $BATCH_SIZE \
    --grad-accum $ACCUM \
    --max-seq-len $MAX_SEQ_LEN \
    --data $DATA_DIR
assert _exit_code == 0, (
    f"학습 실패 (exit {_exit_code}) — 위 트레이스백이 진짜 원인입니다. "
    "여기서 멈추므로 아래 평가 셀이 이어서 도는 일은 없습니다."
)

## 8. 튜닝 모델 평가

튜닝 후에는 **few-shot 없이** 측정합니다. 6번 셀의 베이스라인과 비교하면 파인튜닝 효과가 나옵니다.

In [ ]:
!python finetune/evaluate.py --backend hf --model $MODEL \
    --adapter $OUT/adapter \
    --split test --limit $EVAL_LIMIT --data $DATA_DIR $LAYOUT_FLAG \
    --out $OUT/tuned.json
assert _exit_code == 0, f"튜닝 모델 평가 실패 (exit {_exit_code}) — 위 트레이스백을 확인하세요."

In [ ]:
import json, os

# 어느 평가가 빠졌는지 분명히 알려준다. 그냥 json.load를 하면 FileNotFoundError만
# 떠서 6번 셀이 문제인지 8번 셀이 문제인지 구분이 안 된다.
for path, cell in ((f"{OUT}/baseline.json", "6번(베이스라인)"), (f"{OUT}/tuned.json", "8번(튜닝 후)")):
    assert os.path.exists(path), f"{cell} 평가 결과가 없습니다: {path}"

base = json.load(open(f"{OUT}/baseline.json", encoding="utf-8"))
tuned = json.load(open(f"{OUT}/tuned.json", encoding="utf-8"))

ROWS = [
    ("지표② 상황유형 정확도",  "지표②_상위배치정확도", "situation_accuracy_pct", "90% 이상"),
    ("지표② COP 셀 일치율",   "지표②_상위배치정확도", "cop_cell_match_pct",    "90% 이상"),
    ("지표④ 키워드 정확도",    "지표④_일지정확도",     "keyword_accuracy_pct",  "90% 이상"),
    ("지표④ 누락률",          "지표④_일지정확도",     "omission_rate_pct",     "5% 미만"),
    ("지표④ 일지 kind 정확도", "지표④_일지정확도",     "log_kind_accuracy_pct", "-"),
    ("지표④ ROUGE-L",        "지표④_일지정확도",     "rouge_l_pct",           "-"),
    ("JSON 유효율 (FAST)",    "부가",                "fast_json_valid_pct",   "-"),
    ("지표① FAST 지연(초)",   "지표①_표출지연",       "fast_p50_sec",          "5초 이내"),
]

lines = [f"{'지표':24s} {'튜닝 전':>10s} {'튜닝 후':>10s} {'변화':>10s}   목표", "-" * 74]
for label, group, key, target in ROWS:
    b, t = base[group][key], tuned[group][key]
    lines.append(f"{label:24s} {b:10.2f} {t:10.2f} {t - b:+10.2f}   {target}")
table = "\n".join(lines)
print(table)

# 이 표가 이 노트북의 최종 산출물이다. 뒤 셀이 실패해도 Output에서 다시 볼 수 있게
# 파일로도 남긴다.
with open(f"{OUT}/comparison.txt", "w", encoding="utf-8") as fh:
    fh.write(table + "\n")
print(f"\n저장: {OUT}/comparison.txt")

## 9. 폐쇄망 서빙용 병합 (선택)

어댑터를 베이스에 합쳐 통짜 가중치로 만듭니다. vLLM/Ollama로 부대 내 서버에 올릴 때 쓰며,
앱에서는 사이드바 공급자를 "로컬 서버"로 바꾸기만 하면 코드 수정 없이 연결됩니다.

병합본은 fp16이라 3B 기준 약 6GB다. Kaggle의 노트북 Output 용량 한도가 있으므로, 병합이
끝나면 `checkpoint-*` 중간 산출물은 지우고 `adapter`·`merged`만 남기는 것을 권장한다.

In [ ]:
# 여기는 선택 단계다. 기획서 지표는 8번 셀에서 이미 다 나왔고, 병합본은 폐쇄망
# 서빙용일 뿐이다. 그래서 실패해도 노트북 전체를 실패로 만들지 않는다 — 앞 단계의
# 결과(어댑터, 지표 비교표)는 그대로 Output에 남는다.
!python finetune/train_lora.py --model $MODEL --out $OUT --merge $OUT/merged
if _exit_code == 0:
    !du -sh $OUT/merged
    print("병합 완료 — vLLM/Ollama로 올릴 수 있습니다.")
else:
    print(f"병합 실패 (exit {_exit_code}) — 위 트레이스백을 확인하세요.")
    print("어댑터와 지표 결과는 이미 저장돼 있으므로 학습을 다시 할 필요는 없습니다.")

In [ ]:
# Output 용량을 줄이려면: 중간 체크포인트 삭제 (adapter/merged는 남긴다)
!rm -rf $OUT/checkpoint-*
!du -sh $OUT/*

## 10. 허깅페이스 허브에 자동 업로드 (선택)

9번 셀에서 만든 병합본(`$OUT/merged`)을 학습이 끝나는 즉시 허깅페이스로 올린다. 수동으로
다운로드해서 다시 올리는 단계를 없애기 위한 것이다.

**미리 해 둘 것** (한 번만 하면 이후 실행에서 계속 쓰인다):
1. https://huggingface.co/settings/tokens 에서 **fine-grained 토큰**을 새로 만든다 —
   대상 저장소(위 `HF_REPO_ID`)에 **Write** 권한을 준다.
2. 이 노트북 우측 **Add-ons → Secrets** 에서 **Add secret**, Label은 정확히 `HF_TOKEN`,
   Value는 방금 만든 토큰.
3. 같은 화면에서 **이 노트북에 대해 Attach(켜기)** 를 켠다 — 안 켜면 코드에서 토큰을
   못 읽는다.

병합 단계(9번 셀)와 마찬가지로 실패해도 노트북 전체를 실패로 만들지 않는다 — 학습
결과(어댑터·지표)는 이미 Output에 저장돼 있으므로 업로드만 나중에 수동으로 다시 하면
된다.


In [ ]:
import os

if not os.path.isdir(f"{OUT}/merged"):
    print("병합본이 없어 업로드를 건너뜁니다 (9번 셀이 실패했거나 아직 안 돌았습니다).")
else:
    hf_token = None
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"Kaggle Secrets에서 HF_TOKEN을 못 읽었습니다: {e}")

    if not hf_token:
        print(
            "HF_TOKEN이 없어 업로드를 건너뜁니다. 위 안내대로 Add-ons -> Secrets에서 "
            "write 권한 토큰을 HF_TOKEN이라는 이름으로 추가하고, 이 노트북에 Attach 하세요."
        )
    else:
        try:
            from huggingface_hub import HfApi

            api = HfApi(token=hf_token)
            api.create_repo(HF_REPO_ID, exist_ok=True)
            api.upload_folder(
                folder_path=f"{OUT}/merged",
                repo_id=HF_REPO_ID,
                commit_message=f"Kaggle 자동 업로드 ({MODEL}, {EPOCHS}에폭, "
                               f"{'layout' if LAYOUT else 'classify'})",
            )
            print(f"업로드 완료: https://huggingface.co/{HF_REPO_ID}")
            print(
                "HF Inference Endpoint를 쓰고 있다면 Settings > Advanced > "
                "commit revision을 방금 올린 커밋으로 바꿔야 반영됩니다 "
                "(Pause/Resume만으로는 이전 캐시를 그대로 씁니다)."
            )
        except Exception as e:
            print(f"업로드 실패: {e}")
            print("토큰 권한이 대상 저장소에 Write로 되어 있는지 확인하세요.")
